### JSON 형식 출력 파서
### JsonOutputParser

In [1]:
!pip --version

pip 26.2.1 from D:\hanhwa0902\ex0916\.0916venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_teddynote import logging

logging.langsmith("test0916")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0916


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [5]:
model = ChatOpenAI(temperature=0, model="gpt-5-mini")

In [6]:
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [7]:
question = "지구 온난화의 심각성 대해 알려주세요."

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [13]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

answer = chain.invoke({"question": question})

In [14]:
type(answer)

dict

In [15]:
answer

{'description': '지구 온난화는 대기 중 온실가스(이산화탄소, 메탄 등)의 증가로 지구 평균 기온이 장기적으로 상승하는 현상입니다. 주요 원인은 화석연료 연소, 산림 파괴, 농업 활동 등이며 결과로 해수면 상승, 극한 기후 증가, 생태계 및 농업 피해 등이 발생합니다. 대응책으로는 배출 감축(재생에너지 전환, 에너지 효율 개선), 탄소 흡수 촉진(식림 등), 기후 적응 및 국제 협력이 필요합니다.',
 'hashtags': ['#지구온난화',
  '#기후변화',
  '#온실가스',
  '#이산화탄소',
  '#메탄',
  '#해수면상승',
  '#기상이변',
  '#생물다양성',
  '#재생에너지',
  '#에너지효율',
  '#탄소중립',
  '#적응과완화']}

In [12]:
# Pydantic 을 사용하지 않고 JsonOutputParser 를 사용

question = "지구 온난화에 대해 알려주세요. 온나화에 대한 설명은 'description'에, 관련 키원드는 'hashtags'에 담아주세요."

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

response = chain.invoke({"question": question})

print(response)

{'description': '지구 온난화는 대기 중 온실가스(이산화탄소, 메탄 등)의 증가로 지구 평균 기온이 점진적으로 상승하는 현상입니다. 주요 원인은 화석연료 연소, 산림 파괴, 산업·농업 활동 등으로 인한 탄소 배출이며, 결과적으로 해수면 상승, 극한 기상(폭염·홍수·가뭄)의 빈도 증가, 생태계 파괴 및 인류 건강·식량 안보 위협 등을 초래합니다. 대응은 온실가스 배출 감축(재생에너지 전환, 에너지 효율 개선), 산림 복원, 기후 적응 전략, 국제 협력과 정책 실행을 포함합니다.', 'hashtags': ['#지구온난화', '#기후변화', '#온실가스', '#탄소중립', '#재생에너지', '#해수면상승', '#기후적응', '#탄소배출감소', '#지속가능성', '#산림보호']}
